- This setup is for baseline model investigations
- Default hyperparameters are used, this would be tuned later if selected


In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics,model_selection

In [11]:
train = pd.read_csv("../artifacts/feature_engineering/features_train.csv")
test = pd.read_csv("../artifacts/feature_engineering/features_test.csv")

target = "log_price"

In [12]:
from category_encoders import TargetEncoder


def add_encoded_classes(train, test):
    """
    Apply target encoding to the locality classes
    """
    encoder = TargetEncoder(cols=["loc_class"], smoothing=10.0)

    train["loc_class_enc"] = encoder.fit_transform(
        train["loc_class"], train["log_price"]
    )
    test["loc_class_enc"] = encoder.transform(test["loc_class"])

    train.drop(columns=["loc_class"], inplace=True)
    test.drop(columns=["loc_class"], inplace=True)

    return train, test

train, test = add_encoded_classes(train=train,test=test)

In [13]:
X_train = train.drop(columns=target)
y_train = train[target]

X_test = test.drop(columns=target)
y_test = test[target]

In [14]:
model = RandomForestRegressor()
model = model.fit(X_train, y_train)

In [16]:
def compute_metrics(model, x, y, cv=5):
    preds = model.predict(x)
    score = model.score(x, y)

    scores_cvs = model_selection.cross_val_score(model, x, y, scoring="r2", cv=cv)

    return pd.DataFrame(
        [
            {
                "R2": round(score, 3),
                "mse": round(metrics.mean_squared_error(y, preds), 3),
                "rmse": round(np.sqrt(metrics.mean_squared_error(y, preds)), 3),
                "mae": round(metrics.mean_absolute_error(y, preds), 3),
                "adjusted_r2": round(
                    1 - (1 - score) * (len(y) - 1) / (len(y) - x.shape[1] - 1), 3
                ),
                "cv_score": round(scores_cvs.mean() * 100, 2),
            }
        ]
    )


In [17]:
val_metrics = compute_metrics(model,X_test,y_test)

In [18]:
val_metrics

,R2,mse,rmse,mae,adjusted_r2,cv_score
0,0.825,0.25,0.5,0.369,0.824,81.68


In [20]:
variables = abs(model.feature_importances_)
coef_df = pd.DataFrame(
    {
        "Variable": train.drop(["log_price"], axis=1).columns,
        "Value": variables,
    }
)
n = 20
sorted_df = (
    coef_df.sort_values(by="Value", ascending=False).head(n).sort_values(by="Value")
)
sorted_df


,Variable,Value
10,class_pi,0.002713
20,loc_class_enc,0.003115
6,rot_45_lat,0.003261
4,lat,0.003343
13,loc_trust_score,0.003517
7,rot_45_lng,0.003938
5,lng,0.004537
17,dist_to_cbd,0.005172
19,min_dist_to_hub,0.005850
18,dist_to_east_legon_center,0.006637
